# FT-Transformer

In [1]:
!python -m pip install --upgrade pip setuptools wheel

Looking in indexes: http://mirrors.aliyun.com/pypi/simple


In [2]:
!pip install ipywidgets

Looking in indexes: http://mirrors.aliyun.com/pypi/simple


In [3]:
!pip install rtdl_revisiting_models -q

In [4]:
import pandas as pd
import numpy as np
import os

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from rtdl_revisiting_models import FTTransformer

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
DATA_PATH = "FeatureC_Repeated"
OUTPUT_PATH = "Official_FTTransformer_FeatureC_Results"


os.makedirs(OUTPUT_PATH, exist_ok=True)

N_REPEATS = 10

cuda


In [6]:
def get_feature_cols(df):
    return [
        col for col in df.columns
        if col not in ["userId", "movieId", "label"]
    ]

In [7]:
class TabularDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [8]:
def compute_auc_from_scratch(y_true, y_score):
    y_true = np.array(y_true)
    y_score = np.array(y_score)

    sorted_indices = np.argsort(-y_score)
    y_true_sorted = y_true[sorted_indices]

    pos_count = np.sum(y_true == 1)
    neg_count = np.sum(y_true == 0)

    if pos_count == 0 or neg_count == 0:
        return 0

    tp = 0
    fp = 0

    tpr_list = [0]
    fpr_list = [0]

    for label in y_true_sorted:
        if label == 1:
            tp += 1
        else:
            fp += 1

        tpr_list.append(tp / pos_count)
        fpr_list.append(fp / neg_count)

    auc = 0

    for i in range(1, len(tpr_list)):
        auc += (
            (fpr_list[i] - fpr_list[i - 1])
            * (tpr_list[i] + tpr_list[i - 1])
            / 2
        )

    return auc


def compute_metrics_from_scratch(y_true, y_pred, y_score):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))

    accuracy = (tp + tn) / len(y_true)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0

    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0
    )

    auc = compute_auc_from_scratch(y_true, y_score)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "auc": auc,
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn
    }

In [9]:
def stratified_sample_binary(df, sample_size, random_seed):
    if sample_size >= len(df):
        return df.sample(frac=1, random_state=random_seed).reset_index(drop=True)

    sample_ratio = sample_size / len(df)

    sampled_df = (
        df
        .groupby("label", group_keys=False)
        .apply(
            lambda x: x.sample(
                n=max(1, int(len(x) * sample_ratio)),
                random_state=random_seed
            )
        )
        .sample(frac=1, random_state=random_seed)
        .reset_index(drop=True)
    )

    return sampled_df

In [10]:
def stratified_split_from_scratch(df, label_col, test_ratio=0.2, random_seed=42):
    rng = np.random.default_rng(random_seed)

    train_indices = []
    test_indices = []

    for label_value in df[label_col].unique():
        label_indices = df[df[label_col] == label_value].index.to_numpy()
        rng.shuffle(label_indices)

        test_size = int(len(label_indices) * test_ratio)

        test_indices.extend(label_indices[:test_size])
        train_indices.extend(label_indices[test_size:])

    train_df = df.loc[train_indices].sample(
        frac=1,
        random_state=random_seed
    ).reset_index(drop=True)

    test_df = df.loc[test_indices].sample(
        frac=1,
        random_state=random_seed
    ).reset_index(drop=True)

    return train_df, test_df

In [11]:
def train_official_ft_transformer(
    train_df,
    test_df,
    lr=1e-4,
    weight_decay=1e-5,
    batch_size=4096,
    epochs=3
):
    feature_cols = get_feature_cols(train_df)

    X_train = train_df[feature_cols].values
    y_train = train_df["label"].values

    X_test = test_df[feature_cols].values
    y_test = test_df["label"].values

    scaler = StandardScaler()

    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    train_dataset = TabularDataset(X_train, y_train)
    test_dataset = TabularDataset(X_test, y_test)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False
    )

    model = FTTransformer(
        n_cont_features=len(feature_cols),
        cat_cardinalities=[],
        d_out=1,
        **FTTransformer.get_default_kwargs()
    ).to(device)

    optimizer = model.make_default_optimizer()
    
    # Override default optimizer lr / weight_decay if needed
    for group in optimizer.param_groups:
        group["lr"] = lr
        group["weight_decay"] = weight_decay

    criterion = nn.BCEWithLogitsLoss()

    model.train()

    for epoch in range(epochs):
        total_loss = 0

        for batch_X, batch_y in train_loader:
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)

            optimizer.zero_grad()

            logits = model(batch_X, None).squeeze(1)

            loss = criterion(logits, batch_y)

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch + 1} loss:", round(total_loss, 4))

    model.eval()

    all_preds = []
    all_scores = []
    all_labels = []

    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            batch_X = batch_X.to(device)

            logits = model(batch_X, None).squeeze(1)
            probs = torch.sigmoid(logits)

            preds = (probs >= 0.5).int()

            all_preds.extend(preds.detach().cpu().tolist())
            all_scores.extend(probs.detach().cpu().tolist())
            all_labels.extend(batch_y.detach().cpu().tolist())

    metrics = compute_metrics_from_scratch(
        all_labels,
        all_preds,
        all_scores
    )

    return metrics

In [12]:
TRAIN_SAMPLE_SIZE = 30000
TEST_SAMPLE_SIZE = 10000

LR_VALUES = [5e-5, 1e-4, 5e-4]
WEIGHT_DECAY_VALUES = [1e-5]

N_INNER_REPEATS = 3
VALID_RATIO = 0.2

BATCH_SIZE = 1024
EPOCHS = 3

all_results = []
all_tuning_results = []

for repeat_id in range(1, N_REPEATS + 1):
    print("=" * 60)
    print(f"Outer Repeat {repeat_id:02d}")
    print("=" * 60)

    repeat_folder = os.path.join(
        DATA_PATH,
        f"repeat_{repeat_id:02d}"
    )

    train_path = os.path.join(repeat_folder, "feature_C_train.csv")
    test_path = os.path.join(repeat_folder, "feature_C_test.csv")

    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)

    train_df = stratified_sample_binary(
        train_df,
        sample_size=TRAIN_SAMPLE_SIZE,
        random_seed=42 + repeat_id
    )

    test_df = stratified_sample_binary(
        test_df,
        sample_size=TEST_SAMPLE_SIZE,
        random_seed=100 + repeat_id
    )

    print("Outer train shape:", train_df.shape)
    print("Outer test shape:", test_df.shape)

    best_lr = None
    best_weight_decay = None
    best_mean_val_f1 = -1

    for lr in LR_VALUES:
        for weight_decay in WEIGHT_DECAY_VALUES:

            inner_f1_scores = []

            for inner_id in range(N_INNER_REPEATS):
                inner_train_df, val_df = stratified_split_from_scratch(
                    train_df,
                    label_col="label",
                    test_ratio=VALID_RATIO,
                    random_seed=2000 + repeat_id * 10 + inner_id
                )

                print(
                    f"Trying lr={lr}, weight_decay={weight_decay}, inner={inner_id + 1}"
                )

                val_metrics = train_official_ft_transformer(
                    train_df=inner_train_df,
                    test_df=val_df,
                    lr=lr,
                    weight_decay=weight_decay,
                    batch_size=BATCH_SIZE,
                    epochs=EPOCHS
                )

                inner_f1_scores.append(val_metrics["f1"])

            mean_val_f1 = np.mean(inner_f1_scores)
            std_val_f1 = np.std(inner_f1_scores, ddof=1)

            all_tuning_results.append({
                "outer_repeat": repeat_id,
                "lr": lr,
                "weight_decay": weight_decay,
                "mean_validation_f1": mean_val_f1,
                "std_validation_f1": std_val_f1
            })

            print("Mean validation F1:", round(mean_val_f1, 6))

            if mean_val_f1 > best_mean_val_f1:
                best_mean_val_f1 = mean_val_f1
                best_lr = lr
                best_weight_decay = weight_decay

    print("Best lr:", best_lr)
    print("Best weight_decay:", best_weight_decay)
    print("Best mean validation F1:", best_mean_val_f1)

    test_metrics = train_official_ft_transformer(
        train_df=train_df,
        test_df=test_df,
        lr=best_lr,
        weight_decay=best_weight_decay,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS
    )

    final_result = {
        "repeat": repeat_id,
        "best_lr": best_lr,
        "best_weight_decay": best_weight_decay,
        "batch_size": BATCH_SIZE,
        "best_mean_val_f1": best_mean_val_f1,
        **test_metrics
    }

    all_results.append(final_result)

    print("Accuracy :", round(test_metrics["accuracy"], 4))
    print("Precision:", round(test_metrics["precision"], 4))
    print("Recall   :", round(test_metrics["recall"], 4))
    print("F1       :", round(test_metrics["f1"], 4))
    print("AUC      :", round(test_metrics["auc"], 4))

Outer Repeat 01


/tmp/ipykernel_4741/3738243803.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(
/tmp/ipykernel_4741/3738243803.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


Outer train shape: (29999, 253)
Outer test shape: (9999, 253)
Trying lr=5e-05, weight_decay=1e-05, inner=1
Epoch 1 loss: 15.0897
Epoch 2 loss: 10.0302
Epoch 3 loss: 8.5152
Trying lr=5e-05, weight_decay=1e-05, inner=2
Epoch 1 loss: 13.8786
Epoch 2 loss: 8.7735
Epoch 3 loss: 6.5368
Trying lr=5e-05, weight_decay=1e-05, inner=3
Epoch 1 loss: 14.2532
Epoch 2 loss: 9.0931
Epoch 3 loss: 7.9576
Mean validation F1: 0.888481
Trying lr=0.0001, weight_decay=1e-05, inner=1
Epoch 1 loss: 13.5772
Epoch 2 loss: 8.5341
Epoch 3 loss: 5.9546
Trying lr=0.0001, weight_decay=1e-05, inner=2
Epoch 1 loss: 13.6643
Epoch 2 loss: 8.5617
Epoch 3 loss: 7.0841
Trying lr=0.0001, weight_decay=1e-05, inner=3
Epoch 1 loss: 14.2885
Epoch 2 loss: 8.9095
Epoch 3 loss: 7.1448
Mean validation F1: 0.934943
Trying lr=0.0005, weight_decay=1e-05, inner=1
Epoch 1 loss: 14.6462
Epoch 2 loss: 6.8909
Epoch 3 loss: 3.6247
Trying lr=0.0005, weight_decay=1e-05, inner=2
Epoch 1 loss: 14.3747
Epoch 2 loss: 7.0779
Epoch 3 loss: 3.6966
Tr

/tmp/ipykernel_4741/3738243803.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(
/tmp/ipykernel_4741/3738243803.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


Outer train shape: (29999, 253)
Outer test shape: (9999, 253)
Trying lr=5e-05, weight_decay=1e-05, inner=1
Epoch 1 loss: 13.7169
Epoch 2 loss: 9.1935
Epoch 3 loss: 7.9588
Trying lr=5e-05, weight_decay=1e-05, inner=2
Epoch 1 loss: 14.3709
Epoch 2 loss: 9.3301
Epoch 3 loss: 8.0483
Trying lr=5e-05, weight_decay=1e-05, inner=3
Epoch 1 loss: 14.5183
Epoch 2 loss: 9.3426
Epoch 3 loss: 7.9457
Mean validation F1: 0.879994
Trying lr=0.0001, weight_decay=1e-05, inner=1
Epoch 1 loss: 13.7748
Epoch 2 loss: 8.5802
Epoch 3 loss: 7.1733
Trying lr=0.0001, weight_decay=1e-05, inner=2
Epoch 1 loss: 12.5223
Epoch 2 loss: 8.0656
Epoch 3 loss: 5.6006
Trying lr=0.0001, weight_decay=1e-05, inner=3
Epoch 1 loss: 12.9492
Epoch 2 loss: 8.4137
Epoch 3 loss: 5.8864
Mean validation F1: 0.929656
Trying lr=0.0005, weight_decay=1e-05, inner=1
Epoch 1 loss: 14.3168
Epoch 2 loss: 6.731
Epoch 3 loss: 3.9141
Trying lr=0.0005, weight_decay=1e-05, inner=2
Epoch 1 loss: 14.2754
Epoch 2 loss: 7.269
Epoch 3 loss: 3.944
Trying

...

Outer train shape: (29999, 253)
Outer test shape: (9999, 253)
Trying lr=5e-05, weight_decay=1e-05, inner=1
Epoch 1 loss: 14.5104
Epoch 2 loss: 9.3938
Epoch 3 loss: 7.8661
Trying lr=5e-05, weight_decay=1e-05, inner=2
Epoch 1 loss: 14.8247
Epoch 2 loss: 9.5347
Epoch 3 loss: 8.0164
Trying lr=5e-05, weight_decay=1e-05, inner=3
Epoch 1 loss: 13.7646
Epoch 2 loss: 9.1262
Epoch 3 loss: 7.8533
Mean validation F1: 0.875206
Trying lr=0.0001, weight_decay=1e-05, inner=1
Epoch 1 loss: 14.8168
Epoch 2 loss: 9.1593
Epoch 3 loss: 7.5262
Trying lr=0.0001, weight_decay=1e-05, inner=2
Epoch 1 loss: 13.9121
Epoch 2 loss: 8.7219
Epoch 3 loss: 6.7898
Trying lr=0.0001, weight_decay=1e-05, inner=3
Epoch 1 loss: 13.0235
Epoch 2 loss: 7.7969
Epoch 3 loss: 5.6138
Mean validation F1: 0.921572
Trying lr=0.0005, weight_decay=1e-05, inner=1
Epoch 1 loss: 15.3037
Epoch 2 loss: 7.8303
Epoch 3 loss: 3.7208
Trying lr=0.0005, weight_decay=1e-05, inner=2
Epoch 1 loss: 15.9173
Epoch 2 loss: 8.3544
Epoch 3 loss: 4.1353
Try

/tmp/ipykernel_4741/3738243803.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(
/tmp/ipykernel_4741/3738243803.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


Outer train shape: (29999, 253)
Outer test shape: (9999, 253)
Trying lr=5e-05, weight_decay=1e-05, inner=1
Epoch 1 loss: 13.5937
Epoch 2 loss: 9.0304
Epoch 3 loss: 6.4557
Trying lr=5e-05, weight_decay=1e-05, inner=2
Epoch 1 loss: 15.2949
Epoch 2 loss: 9.8029
Epoch 3 loss: 8.0852
Trying lr=5e-05, weight_decay=1e-05, inner=3
Epoch 1 loss: 15.3289
Epoch 2 loss: 9.7458
Epoch 3 loss: 7.9976
Mean validation F1: 0.888452
Trying lr=0.0001, weight_decay=1e-05, inner=1
Epoch 1 loss: 12.0177
Epoch 2 loss: 7.6777
Epoch 3 loss: 5.3539
Trying lr=0.0001, weight_decay=1e-05, inner=2
Epoch 1 loss: 12.5946
Epoch 2 loss: 8.2819
Epoch 3 loss: 6.2312
Trying lr=0.0001, weight_decay=1e-05, inner=3
Epoch 1 loss: 13.8836
Epoch 2 loss: 8.7725
Epoch 3 loss: 6.3322
Mean validation F1: 0.928279
Trying lr=0.0005, weight_decay=1e-05, inner=1
Epoch 1 loss: 13.2299
Epoch 2 loss: 6.9378
Epoch 3 loss: 3.7678
Trying lr=0.0005, weight_decay=1e-05, inner=2
Epoch 1 loss: 13.9621
Epoch 2 loss: 7.2944
Epoch 3 loss: 3.9285
Try

/tmp/ipykernel_4741/3738243803.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(
/tmp/ipykernel_4741/3738243803.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


Outer train shape: (29999, 253)
Outer test shape: (9999, 253)
Trying lr=5e-05, weight_decay=1e-05, inner=1
Epoch 1 loss: 14.1153
Epoch 2 loss: 9.0494
Epoch 3 loss: 7.3745
Trying lr=5e-05, weight_decay=1e-05, inner=2
Epoch 1 loss: 14.8569
Epoch 2 loss: 9.1357
Epoch 3 loss: 7.7734
Trying lr=5e-05, weight_decay=1e-05, inner=3
Epoch 1 loss: 14.9774
Epoch 2 loss: 9.5196
Epoch 3 loss: 7.9489
Mean validation F1: 0.882025
Trying lr=0.0001, weight_decay=1e-05, inner=1
Epoch 1 loss: 13.691
Epoch 2 loss: 8.7803
Epoch 3 loss: 7.1586
Trying lr=0.0001, weight_decay=1e-05, inner=2
Epoch 1 loss: 13.7667
Epoch 2 loss: 8.203
Epoch 3 loss: 6.379
Trying lr=0.0001, weight_decay=1e-05, inner=3
Epoch 1 loss: 13.2664
Epoch 2 loss: 8.5546
Epoch 3 loss: 6.3697
Mean validation F1: 0.913215
Trying lr=0.0005, weight_decay=1e-05, inner=1
Epoch 1 loss: 12.9602
Epoch 2 loss: 6.2672
Epoch 3 loss: 4.0624
Trying lr=0.0005, weight_decay=1e-05, inner=2
Epoch 1 loss: 13.5771
Epoch 2 loss: 6.7298
Epoch 3 loss: 3.9719
Trying

In [13]:
results_df = pd.DataFrame(all_results)
tuning_results_df = pd.DataFrame(all_tuning_results)

results_path = os.path.join(
    OUTPUT_PATH,
    "FTTransformer_FeatureC_repeated_results.csv"
)

tuning_path = os.path.join(
    OUTPUT_PATH,
    "FTTransformer_FeatureC_tuning_results.csv"
)

results_df.to_csv(results_path, index=False, encoding="utf-8-sig")
tuning_results_df.to_csv(tuning_path, index=False, encoding="utf-8-sig")

print("Saved repeated test results to:")
print(results_path)

print("Saved tuning results to:")
print(tuning_path)

results_df

Saved repeated test results to:
Official_FTTransformer_FeatureC_Results/FTTransformer_FeatureC_repeated_results.csv
Saved tuning results to:
Official_FTTransformer_FeatureC_Results/FTTransformer_FeatureC_tuning_results.csv


,repeat,best_lr,best_weight_decay,batch_size,best_mean_val_f1,accuracy,precision,recall,f1,auc,tp,tn,fp,fn
0,1,0.0005,0.00001,1024,0.944880,0.625263,0.650385,0.540925,0.590626,0.650643,2703,3549,1453,2294
1,2,0.0005,0.00001,1024,0.940293,0.625463,0.634335,0.591555,0.612198,0.658074,2956,3298,1704,2041
2,3,0.0005,0.00001,1024,0.941773,0.638964,0.655948,0.583750,0.617747,0.672804,2917,3472,1530,2080
3,4,0.0005,0.00001,1024,0.941642,0.615262,0.659192,0.476486,0.553142,0.665585,2381,3771,1231,2616
4,5,0.0005,0.00001,1024,0.943480,0.622362,0.652816,0.521913,0.580071,0.655688,2608,3615,1387,2389
5,6,0.0005,0.00001,1024,0.942488,0.629963,0.644464,0.578947,0.609952,0.658274,2893,3406,1596,2104
6,7,0.0005,0.00001,1024,0.946478,0.623662,0.651374,0.531319,0.585253,0.657016,2655,3581,1421,2342
7,8,0.0005,0.00001,1024,0.947973,0.642264,0.638835,0.653792,0.646227,0.678477,3267,3155,1847,1730
8,9,0.0005,0.00001,1024,0.947469,0.629463,0.644132,0.577747,0.609136,0.667550,2887,3407,1595,2110
9,10,0.0005,0.00001,1024,0.939141,0.643464,0.652730,0.612367,0.631905,0.674840,3060,3374,1628,1937


In [14]:
summary_records = []

for metric in ["accuracy", "precision", "recall", "f1", "auc"]:
    values = results_df[metric].values

    summary_records.append({
        "metric": metric,
        "mean": np.mean(values),
        "std": np.std(values, ddof=1),
        "standard_error": np.std(values, ddof=1) / np.sqrt(len(values))
    })

summary_df = pd.DataFrame(summary_records)

summary_path = os.path.join(
    OUTPUT_PATH,
    "FTTransformer_FeatureC_summary.csv"
)

summary_df.to_csv(
    summary_path,
    index=False,
    encoding="utf-8-sig"
)

summary_df

,metric,mean,std,standard_error
0,accuracy,0.629613,0.009247,0.002924
1,precision,0.648421,0.007800,0.002466
2,recall,0.566880,0.050455,0.015955
3,f1,0.603626,0.027014,0.008543
4,auc,0.663895,0.009332,0.002951


In [15]:
# Best learning rate frequency for FT-Transformer

best_lr_frequency = (
    results_df["best_lr"]
    .value_counts()
    .reset_index()
)

best_lr_frequency.columns = ["learning_rate", "frequency"]

best_lr_frequency_path = os.path.join(
    OUTPUT_PATH,
    "FTTransformer_FeatureC_best_lr_frequency.csv"
)

best_lr_frequency.to_csv(
    best_lr_frequency_path,
    index=False,
    encoding="utf-8-sig"
)


best_lr_frequency

,learning_rate,frequency
0,0.0005,10
